In [ ]:
from notebook.services.config import ConfigManager
cm = ConfigManager()
cm.update('livereveal', {'width': 1920, 'height': 1080, 'scroll': True})

# Week 13: Wednesday, AST 5011: Astrophysical Systems

## Galaxy Evolution Across Cosmic Time

### Michael Coughlin

References: Madau & Dickinson 2014 (ARA&A); Mo, van den Bosch & White, Ch. 15; Somerville & Davé 2015 review. (Beyond CFN.)

With material from Benedikt Diemer (UMD).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from colossus.cosmology import cosmology

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

cosmo = cosmology.setCosmology('planck18')

def sfrd_madau14(z):
    """Cosmic SFRD (Msun/yr/Mpc^3) from Madau & Dickinson (2014)."""
    return 0.015 * (1 + z)**2.7 / (1.0 + ((1 + z) / 2.9)**5.6)

def sizeEvolution(z, M_star=1e10, galaxy_type='late'):
    """Median effective radius (kpc) vs redshift at fixed stellar mass (van der Wel+2014)."""
    log_m = np.log10(M_star / 5e10)
    if galaxy_type == 'late':
        R_0 = 6.0 * 10**(0.22 * log_m)
        beta = -0.75
    elif galaxy_type == 'early':
        R_0 = 4.0 * 10**(0.60 * log_m)
        beta = -1.48
    else:
        raise ValueError(f"galaxy_type must be 'early' or 'late', got '{galaxy_type}'")
    return R_0 * (1 + z)**beta

def quenchedFraction(z, log_Mstar):
    """Approximate quenched fraction vs redshift and stellar mass (Muzzin+2013 / Tomczak+2014)."""
    log_M_q = 10.2 + 0.5 * z
    delta = 0.8
    f_max = np.clip(0.85 - 0.15 * z, 0.1, 0.85)
    return f_max / (1.0 + np.exp(-(log_Mstar - log_M_q) / delta))

def ssfr_mainSequence(z, log_Mstar):
    """Specific SFR (yr^-1) on the star-forming main sequence (Speagle+2014)."""
    t = cosmo.age(z)
    log_sfr = (0.84 - 0.026 * t) * log_Mstar - (6.51 - 0.11 * t)
    return 10**(log_sfr - log_Mstar)

## Galaxy Evolution: The Big Picture

Over the past several lectures we have built up the theoretical framework for galaxy formation: dark matter halos, gas cooling, star formation, feedback, mergers, and scaling relations. Today we ask: how does the galaxy population as a whole evolve with cosmic time?

![Cosmic Timeline](figures/cosmic_timeline.png)

The observational evidence comes from galaxy surveys at multiple redshifts. At $z \sim 0$, SDSS provides spectra and photometry for millions of galaxies. At $z \sim 0.5$-$3$, HST surveys like CANDELS, GOODS, and COSMOS resolve galaxy morphologies and measure photometric redshifts for hundreds of thousands of galaxies. At $z > 6$, JWST is now discovering galaxies in the first billion years of cosmic history.

These surveys reveal four fundamental trends:
- The cosmic star formation rate density rises to a peak at $z \sim 2$ and declines to the present
- The cumulative stellar mass budget tracks this integral, with half of all stars formed by $z \sim 1.5$
- Galaxies were physically smaller at high redshift, especially quiescent systems
- The fraction of quenched galaxies grows with time and with mass (downsizing)

Each of these trends encodes the physics we have studied: halo assembly, gas accretion and cooling, star formation, AGN and stellar feedback, and mergers. This lecture is the observational scorecard for the entire theoretical framework.

## The Cosmic Star Formation Rate Density

The most fundamental measure of galaxy evolution is the cosmic star formation rate density (SFRD): the total mass of stars formed per year per comoving volume, $\psi(z)$, as a function of redshift.

Measuring the SFRD requires two complementary tracers:
- UV continuum: directly traces emission from young, massive O and B stars (lifetimes $< 100$ Myr). Accessible from the ground at $z > 1$ (where rest-frame UV shifts into the optical). However, UV is strongly attenuated by dust.
- Infrared emission: dust absorbs UV light and re-emits it thermally at $\lambda \sim 8$-$1000\,\mu$m. Herschel and Spitzer measured this component. At cosmic noon ($z \sim 2$), roughly half of all star formation is obscured by dust.

Madau & Dickinson (2014) compiled decades of UV and IR measurements across $0 < z < 8$ and fit the total (dust-corrected) SFRD:

$$\psi(z) = 0.015\,\frac{(1+z)^{2.7}}{1 + \left(\frac{1+z}{2.9}\right)^{5.6}}\quad M_\odot\,\text{yr}^{-1}\,\text{Mpc}^{-3}$$

The shape reflects the competition between two effects:
- At early times ($z > 3$), most dark matter halos are still too low-mass to accrete and cool gas efficiently
- At late times ($z < 1$), gas supplies are depleted, massive galaxies are quenched by AGN feedback, and the cosmic gas density has dropped

The peak at $z \sim 1.5$-$2.5$ -- cosmic noon -- is the sweet spot where both conditions are favorable. The Universe was forming stars about 10 times faster at cosmic noon than it does today.

## Exercise 1: The Madau-Dickinson Plot

1. Plot the cosmic SFRD from $z = 0$ to $z = 10$ using the Madau & Dickinson (2014) fitting function
2. Overplot the SFRD on both a redshift axis and a lookback time axis
3. Compute the cumulative stellar mass density by integrating the SFRD (with a recycling fraction $R = 0.27$ for a Chabrier IMF)

In [ ]:
# Exercise 1: The Madau-Dickinson Plot

z_arr = np.linspace(0, 10, 500)
t_arr = cosmo.age(z_arr)  # Gyr
t_lookback = cosmo.age(0.0) - t_arr  # lookback time

# FILL IN: compute SFRD
sfrd =  # FILL IN

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: SFRD vs redshift
ax1 = axes[0]
ax1.semilogy(z_arr, sfrd, 'b-', lw=2)
ax1.set_xlabel('Redshift $z$')
ax1.set_ylabel(r'$\psi(z)$ ($M_\odot$ yr$^{-1}$ Mpc$^{-3}$)')
ax1.set_xlim(0, 10)
ax1.set_ylim(1e-3, 0.3)
ax1.axvspan(1.5, 2.5, alpha=0.1, color='red')
ax1.text(2.0, 0.15, 'Cosmic Noon', ha='center', fontsize=9, color='red')
ax1.set_title('Cosmic Star Formation Rate Density')

# Right: SFRD vs lookback time
ax2 = axes[1]
ax2.semilogy(t_lookback, sfrd, 'b-', lw=2)
ax2.set_xlabel('Lookback time (Gyr)')
ax2.set_ylabel(r'$\psi$ ($M_\odot$ yr$^{-1}$ Mpc$^{-3}$)')
ax2.set_xlim(0, 13.5)
ax2.set_ylim(1e-3, 0.3)
ax2.set_title('SFRD vs Lookback Time')

plt.tight_layout()
plt.show()

# SFRD at key epochs
for z_val in [0, 1, 2, 3, 6]:
    print(f'SFRD at z={z_val}: {sfrd_madau14(z_val):.4f} Msun/yr/Mpc^3')

# Ratio of peak to present
print(f'\nPeak/present ratio: {sfrd_madau14(2.0) / sfrd_madau14(0.0):.1f}x')

## Demonstration: The Cosmic Stellar Mass Budget

By integrating the SFRD over time (accounting for mass returned to the ISM via stellar evolution), we can compute the cumulative stellar mass density:

$$\rho_*(z) = (1 - R) \int_{t(z_{\rm ini})}^{t(z)} \psi(z')\,dt'$$

where $R \approx 0.27$ is the recycling fraction for a Chabrier IMF -- the fraction of stellar mass returned to the ISM through winds and supernovae over a Hubble time. The factor $(1 - R)$ accounts for the fact that not all mass that forms stars remains locked in stars; about 27% is returned.

This integral should match the stellar mass density measured directly by adding up the stellar masses of all galaxies at each redshift (from SED fitting). Agreement at the factor-of-2 level validates both the SFRD measurements and the stellar mass estimates. Persistent discrepancies have historically pointed to:
- Missing dust-obscured star formation (resolved by Herschel at $z \sim 1$-$3$)
- Systematic errors in SED-derived stellar masses (the "outshining" problem, where young stars mask the older population)
- Possible IMF variations with redshift or environment

In [ ]:
# Stellar mass density from integrating the SFRD

z_smd = np.linspace(0, 8, 100)
R = 0.27  # recycling fraction (Chabrier IMF)

# Integrate SFRD from z=20 to each z
rho_star = np.zeros_like(z_smd)
for i, z in enumerate(z_smd):
    z_int = np.linspace(20.0, z, 500)
    t_int = cosmo.age(z_int) * 1e9  # yr
    sfrd_int = sfrd_madau14(z_int)
    rho_star[i] = (1 - R) * np.abs(np.trapz(sfrd_int, t_int))

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.semilogy(z_smd, rho_star, 'b-', lw=2, label='Integrated SFRD')
ax.set_xlabel('Redshift $z$')
ax.set_ylabel(r'$\rho_*$ ($M_\odot$ Mpc$^{-3}$)')
ax.set_xlim(0, 8)
ax.set_ylim(1e5, 1e9)
ax.legend(fontsize=10)
ax.set_title('Cosmic Stellar Mass Density')
ax.invert_xaxis()
plt.tight_layout()
plt.show()

print(f'Stellar mass density at z=0: {rho_star[0]:.2e} Msun/Mpc^3')
print(f'Half of all stars formed by z ~ {z_smd[np.argmin(np.abs(rho_star - 0.5*rho_star[0]))]:.1f}')

## Galaxy Size Evolution

One of the most dramatic discoveries of the HST era: galaxies at high redshift are much smaller than their local counterparts at the same stellar mass. This was first established by Ferguson et al. (2004) and Trujillo et al. (2006), and quantified systematically by van der Wel et al. (2014) using ~30,000 galaxies from the CANDELS HST survey.

van der Wel et al. (2014) parameterized the median effective radius at fixed stellar mass as:

$$R_e(z) = R_0\,(1+z)^\beta$$

with $\beta \approx -0.75$ for star-forming (late-type) galaxies and $\beta \approx -1.48$ for quiescent (early-type) galaxies. The difference in $\beta$ between the two populations is physically significant:

Star-forming galaxies grow through a combination of gas accretion (which builds the disk from the inside out), in-situ star formation, and minor mergers. This produces moderate size growth ($\beta \approx -0.75$).

Quiescent galaxies grow primarily through dry (gas-poor) minor mergers after they have quenched. Accreted satellite stars are deposited preferentially in the outskirts (large radii), "puffing up" the galaxy -- increasing $R_e$ without adding much mass to the center or forming new stars. This produces rapid size growth ($\beta \approx -1.5$).

At $z \sim 2$, massive quiescent galaxies ($M_* \sim 10^{11}\,M_\odot$) have $R_e \sim 1$-$2$ kpc -- about 5 times smaller than their local counterparts at $R_e \sim 5$-$8$ kpc. These compact massive galaxies are sometimes called "red nuggets." A few rare examples have been found in the local Universe as well, having somehow avoided mergers for 10 billion years.

## Exercise 2: Galaxy Size Evolution

1. Plot the median effective radius vs redshift for star-forming and quiescent galaxies at $M_* = 10^{10}$ and $10^{11}\,M_\odot$
2. At what redshift is a quiescent galaxy half the size of its $z = 0$ counterpart?
3. Compare the size evolution rates: why do quiescent galaxies evolve faster?

In [ ]:
# Exercise 2: Galaxy size evolution

z_size = np.linspace(0, 3, 100)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# Left: R_e vs z for different masses
ax1 = axes[0]
for log_m, ls in [(10, '-'), (11, '--')]:
    M = 10**log_m
    R_late =  # FILL IN: size evolution for late-type
    R_early = # FILL IN: size evolution for early-type
    ax1.plot(z_size, R_late, ls=ls, color='C0', lw=2,
             label=f'Star-forming, $10^{{{log_m}}}\,M_\odot$')
    ax1.plot(z_size, R_early, ls=ls, color='C1', lw=2,
             label=f'Quiescent, $10^{{{log_m}}}\,M_\odot$')

ax1.set_xlabel('Redshift $z$')
ax1.set_ylabel(r'$R_e$ (kpc)')
ax1.set_xlim(0, 3)
ax1.legend(fontsize=7)
ax1.set_title('Size Evolution')

# Right: ratio R_e(z) / R_e(0)
ax2 = axes[1]
for gtype, color, label in [('late', 'C0', 'Star-forming'), ('early', 'C1', 'Quiescent')]:
    R_z = sizeEvolution(z_size, M_star=1e11, galaxy_type=gtype)
    R_0 = sizeEvolution(0.0, M_star=1e11, galaxy_type=gtype)
    ax2.plot(z_size, R_z / R_0, color=color, lw=2, label=label)

ax2.axhline(0.5, ls='--', color='gray', lw=0.8)
ax2.text(2.5, 0.53, 'Half size', fontsize=9, color='gray')
ax2.set_xlabel('Redshift $z$')
ax2.set_ylabel(r'$R_e(z) / R_e(0)$')
ax2.set_xlim(0, 3)
ax2.set_ylim(0, 1.1)
ax2.legend(fontsize=9)
ax2.set_title(r'Size Ratio ($M_* = 10^{11}\,M_\odot$)')

plt.tight_layout()
plt.show()

# FILL IN: find z where quiescent galaxies are half their z=0 size

## Demonstration: The Mass-Size Relation Across Redshift

The mass-size relation -- the correlation between stellar mass and effective radius -- is one of the most informative projections of galaxy structure. At fixed stellar mass, star-forming galaxies are systematically larger than quiescent galaxies at all redshifts. The offset between the two populations grows at higher redshift, reflecting their different growth channels.

The surface mass density within the effective radius, $\Sigma_e = M_* / (2\pi R_e^2)$, is a useful diagnostic. Quiescent galaxies at $z \sim 2$ have $\Sigma_e \sim 10^{10}\,M_\odot\,\text{kpc}^{-2}$ -- comparable to the densest stellar systems in the local Universe (globular clusters and nuclear star clusters). These extreme densities are consistent with formation through gas-rich, dissipative processes (wet mergers or violent disk instabilities) that funnel gas to the center.

In [ ]:
# Mass-size relation at multiple redshifts

log_Mstar_ms = np.linspace(9, 11.5, 100)
M_star_ms = 10**log_Mstar_ms
z_epochs = [0.0, 0.5, 1.0, 2.0]
colors_z = ['black', 'C0', 'C1', 'C2']

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: mass-size relation
ax1 = axes[0]
for z_val, color in zip(z_epochs, colors_z):
    R_late = sizeEvolution(z_val, M_star=M_star_ms, galaxy_type='late')
    R_early = sizeEvolution(z_val, M_star=M_star_ms, galaxy_type='early')
    ax1.plot(log_Mstar_ms, np.log10(R_late), '-', color=color, lw=2, alpha=0.7)
    ax1.plot(log_Mstar_ms, np.log10(R_early), '--', color=color, lw=2, alpha=0.7,
             label=f'$z = {z_val}$' if z_val == z_epochs[0] or z_val == z_epochs[-1] else None)

# Add legend entries for line styles
ax1.plot([], [], 'k-', lw=2, label='Star-forming')
ax1.plot([], [], 'k--', lw=2, label='Quiescent')
ax1.set_xlabel(r'$\log_{10}\,M_*\,(M_\odot)$')
ax1.set_ylabel(r'$\log_{10}\,R_e$ (kpc)')
ax1.legend(fontsize=8)
ax1.set_title('Mass-Size Relation')

# Right: surface mass density
ax2 = axes[1]
for z_val, color in zip(z_epochs, colors_z):
    R_early = sizeEvolution(z_val, M_star=M_star_ms, galaxy_type='early')
    Sigma_e = M_star_ms / (2 * np.pi * R_early**2)
    ax2.plot(log_Mstar_ms, np.log10(Sigma_e), color=color, lw=2, label=f'$z = {z_val}$')

ax2.axhline(10, ls=':', color='gray', lw=0.8)
ax2.text(9.2, 10.1, r'$\Sigma_e = 10^{10}\,M_\odot\,\mathrm{kpc}^{-2}$', fontsize=8, color='gray')
ax2.set_xlabel(r'$\log_{10}\,M_*\,(M_\odot)$')
ax2.set_ylabel(r'$\log_{10}\,\Sigma_e\,(M_\odot\,\mathrm{kpc}^{-2})$')
ax2.legend(fontsize=9)
ax2.set_title('Surface Mass Density (Quiescent)')

plt.tight_layout()
plt.show()

## Downsizing and the Quenching of Star Formation

One of the most important empirical results in galaxy evolution is downsizing (Cowie et al. 1996): the most massive galaxies formed their stars earliest and quenched first, while low-mass galaxies continue forming stars to the present day.

![Downsizing](figures/downsizing.png)

This seems paradoxical in a hierarchical universe where massive halos assemble last. The resolution is that "assembly" (of the dark matter halo, via mergers) and "formation" (of the stars) are different processes. The stars in a massive elliptical may have formed at $z \sim 3$ in several smaller progenitor galaxies, which subsequently merged (without forming many new stars) to assemble the massive galaxy by $z \sim 1$. The stars formed early even though the galaxy assembled late.

Evidence for downsizing comes from two complementary approaches:
- Archaeological (fossil record): spectroscopic analysis of local galaxies reveals that massive ellipticals have old, alpha-enhanced stellar populations (Thomas et al. 2005), indicating short, intense star formation episodes at early times. Low-mass galaxies have younger, more extended star formation histories.
- Lookback (direct observation): galaxy surveys at $z \sim 1$-$3$ show that the massive end of the star-forming main sequence shuts down first, with the characteristic mass of star-forming galaxies decreasing with time.

The quenched (red/passive) fraction of galaxies increases with stellar mass and decreases with redshift. At $z = 0$, nearly all galaxies above $M_* \sim 10^{11}\,M_\odot$ are quenched, while most below $10^{10}\,M_\odot$ are still forming stars. At $z = 2$, even massive galaxies had significant star formation.

The physical drivers of quenching connect directly to the physics from earlier lectures:
- AGN feedback (Lecture 20): dominant for massive centrals in halos with $M_h > 10^{12}\,M_\odot$. Radio-mode jets maintain the quenched state by heating the hot gas halo.
- Environmental effects (Lecture 21): satellite quenching via strangulation, ram pressure stripping, and tidal effects. Operates after a delay of $\sim 2$-$4$ Gyr after infall.
- Halo mass quenching: in halos above $\sim 10^{12}\,M_\odot$, the virial shock heats infalling gas to the virial temperature, preventing efficient cooling and cutting off the cold gas supply. This is the "cooling cutoff" from Lecture 19.

## Exercise 3: The Quenched Fraction and Specific SFR

1. Plot the quenched fraction as a function of stellar mass at $z = 0, 0.5, 1, 2$
2. Plot the specific SFR (sSFR = SFR/$M_*$) on the star-forming main sequence at the same redshifts
3. At what stellar mass does the quenched fraction exceed 50% at each redshift?

In [ ]:
# Exercise 3: Quenched fraction and sSFR evolution

log_Mstar = np.linspace(9, 12, 100)
z_vals = [0.0, 0.5, 1.0, 2.0]
colors = ['black', 'C0', 'C1', 'C2']

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# Left: quenched fraction vs stellar mass
ax1 = axes[0]
for z_val, color in zip(z_vals, colors):
    f_q =  # FILL IN: quenched fraction
    ax1.plot(log_Mstar, f_q, color=color, lw=2, label=f'$z = {z_val}$')

ax1.axhline(0.5, ls='--', color='gray', lw=0.8)
ax1.set_xlabel(r'$\log_{10}\,M_*\,(M_\odot)$')
ax1.set_ylabel('Quenched fraction')
ax1.set_xlim(9, 12)
ax1.set_ylim(0, 1)
ax1.legend(fontsize=9)
ax1.set_title('Quenched Fraction vs Stellar Mass')

# Right: sSFR on the main sequence
ax2 = axes[1]
for z_val, color in zip(z_vals, colors):
    ssfr =  # FILL IN: specific SFR
    ax2.plot(log_Mstar, np.log10(ssfr), color=color, lw=2, label=f'$z = {z_val}$')

ax2.axhline(-11, ls='--', color='gray', lw=0.8)
ax2.text(11.5, -10.8, 'Quenching threshold', fontsize=8, color='gray')
ax2.set_xlabel(r'$\log_{10}\,M_*\,(M_\odot)$')
ax2.set_ylabel(r'$\log_{10}$ sSFR (yr$^{-1}$)')
ax2.set_xlim(9, 12)
ax2.set_ylim(-12, -8)
ax2.legend(fontsize=9)
ax2.set_title('Specific SFR on the Main Sequence')

plt.tight_layout()
plt.show()

# FILL IN: find the mass where f_q = 0.5 at each redshift

## Demonstration: JWST and the High-Redshift Frontier

JWST, launched in December 2021, has opened a transformative window on the first billion years of galaxy evolution. Its 6.5-meter mirror and infrared sensitivity (0.6-28 $\mu$m) allow it to detect rest-frame UV and optical light from galaxies at $z > 6$, where ground-based telescopes are blind.

Key discoveries from the first three years of JWST observations:

Unexpectedly abundant bright galaxies at $z > 10$: the UV luminosity function at $z = 10$-$13$ shows 3-10 times more bright galaxies than most pre-launch models predicted. Several spectroscopically confirmed galaxies now exist at $z > 13$, less than 300 Myr after the Big Bang.

Overmassive galaxies: some galaxies at $z \sim 7$-$10$ appear to have stellar masses of $10^9$-$10^{10}\,M_\odot$, which would require star formation efficiencies approaching or exceeding the cosmic baryon fraction in their halos -- uncomfortably high given the expected impact of feedback.

Early AGN activity: JWST has identified broad-line AGN at $z > 10$, confirming that supermassive black holes began growing very early. Some of these BHs appear overmassive relative to their host galaxies compared to local scaling relations (Lecture 20), suggesting different growth pathways at early times.

Mature stellar populations at intermediate redshifts: at $z \sim 4$-$6$, JWST finds galaxies with Balmer breaks, indicating stellar populations older than $\sim 500$ Myr, and surprisingly regular morphologies (disks, bars) that were not expected to form so early.

Possible explanations for the high-$z$ excess include: (1) higher star formation efficiency in early halos with lower metallicity gas, (2) less dust attenuation at very early times, (3) bursty star formation histories that make galaxies temporarily brighter than their time-averaged SFR would suggest, or (4) a top-heavy IMF at early times that produces more UV light per unit mass formed. These are not mutually exclusive, and JWST spectroscopy (e.g., with NIRSpec) is actively testing each scenario.

In [ ]:
# JWST vs pre-JWST: SFRD at high redshift

z_high = np.linspace(0, 15, 500)
sfrd_md14 = sfrd_madau14(z_high)

# Schematic: JWST observations suggest higher SFRD at z > 8
sfrd_jwst = sfrd_md14.copy()
mask_high = z_high > 7
sfrd_jwst[mask_high] = sfrd_md14[mask_high] * (1 + 2 * (z_high[mask_high] - 7) / 5)

fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogy(z_high, sfrd_md14, 'b-', lw=2, label='Madau & Dickinson (2014)')
ax.semilogy(z_high[mask_high], sfrd_jwst[mask_high], 'r--', lw=2,
            label='JWST hints (schematic)')
ax.fill_between(z_high[mask_high],
                sfrd_md14[mask_high] * 0.5,
                sfrd_jwst[mask_high] * 2,
                alpha=0.1, color='red')

ax.axvspan(8, 15, alpha=0.05, color='gold')
ax.text(11, 0.05, 'JWST frontier', fontsize=11, ha='center', color='goldenrod')

ax.set_xlabel('Redshift $z$')
ax.set_ylabel(r'$\psi(z)$ ($M_\odot$ yr$^{-1}$ Mpc$^{-3}$)')
ax.set_xlim(0, 15)
ax.set_ylim(1e-4, 0.5)
ax.legend(fontsize=10)
ax.set_title('Cosmic SFRD: Pre-JWST vs JWST Era')
plt.tight_layout()
plt.show()

print('JWST key result: the UV luminosity function at z > 10 shows')
print('3-10x more bright galaxies than predicted by most pre-launch models.')
print('Possible explanations: higher star formation efficiency, less dust,')
print('bursty SFHs, or top-heavy IMF at early times.')

## Connecting the Course: From Fluctuations to the Observed Universe

This lecture brings our journey full circle. Here is how each major topic we covered connects to the observations discussed today:

Lectures 14-15 (Cosmology and density fluctuations): the primordial power spectrum sets the initial conditions for structure formation. The shape of the halo mass function -- which determines how many galaxies of each mass exist at each epoch -- is a direct consequence of the power spectrum processed through the transfer function and growth factor.

Lecture 16 (Dark matter halos): the NFW profile and concentration-mass relation determine the gravitational potential wells that galaxies form in. The halo mass function evolution (which we saw in Lecture 21) directly predicts the hierarchical assembly of structure -- small halos first, clusters last.

Lectures 17-18 (Gas accretion, cooling, stellar populations): the cooling function and virial temperature determine which halos can form stars efficiently. The IMF determines the recycling fraction $R$ that enters the stellar mass budget calculation, and the mass-to-light ratios needed to convert observed luminosities to stellar masses.

Lecture 19 (Semi-analytic model): the SAM we built encodes all of this physics in a single set of coupled ODEs. The SHMR it produces is the bridge between the dark matter halo mass function and the galaxy stellar mass function -- and the SFRD is essentially the derivative of the integral of the stellar mass function over time.

Lectures 20-21 (AGN feedback, mergers, satellites): AGN feedback produces the quenching of massive galaxies (the high-mass end of the quenched fraction plot). Mergers drive the size evolution of quiescent galaxies. Environmental quenching produces the satellite contribution to the quenched fraction.

Lecture 22 (Scaling relations): the Tully-Fisher and Fundamental Plane relations are the $z = 0$ endpoint of the size, mass, and velocity evolution we tracked today. Their tightness constrains how uniform the galaxy formation process is at fixed halo mass.

## Summary

1. The cosmic SFRD peaks at $z \sim 2$ (cosmic noon) at roughly 10x the present-day rate. UV and IR tracers are both needed to account for dust-obscured star formation. The Madau-Dickinson plot is one of the most fundamental observational constraints on galaxy formation.

2. Integrating the SFRD (with recycling fraction $R = 0.27$) gives the cumulative stellar mass density, which must match directly measured values from galaxy surveys. About half of all stars were formed by $z \sim 1.5$.

3. Galaxies at high redshift are significantly smaller than local counterparts at fixed stellar mass. Quiescent galaxies evolve especially fast ($R_e \propto (1+z)^{-1.5}$), growing primarily through dry minor mergers that deposit accreted stars in the outskirts. Compact massive galaxies ("red nuggets") at $z \sim 2$ have surface mass densities rivaling globular clusters.

4. Downsizing: massive galaxies formed their stars earliest and quenched first, despite assembling their halos last. The quenched fraction increases with mass and decreases with redshift, driven by AGN feedback (massive centrals), halo mass quenching (virial shocks), and environmental processes (satellites).

5. The specific SFR on the star-forming main sequence increases with redshift -- galaxies at $z \sim 2$ formed stars $\sim 10\times$ faster per unit mass than similar galaxies today. The normalization of the main sequence tracks the SFRD.

6. JWST is revealing unexpectedly abundant and luminous galaxies at $z > 10$, pushing the boundaries of our models. Possible explanations include higher star formation efficiency, reduced dust, bursty SFHs, or IMF variations at early times.

Next lecture we close out the semester with two topics the galaxy-evolution panorama left open: **chemical evolution** (how galaxies built up their heavy elements) and **cosmic reionization** (how the Universe transitioned from the neutral dark ages to the ionized IGM we observe today). Both pick up the SFRD we worked with here and push it in new directions.